<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/22_conv2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn.functional as F

In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
  if padding != 0:
    pads = [padding] * 4
    x = F.pad(x, pads, "constant", 0)
  assert len(x.shape) == 4
  assert len(weight.shape) == 4
  B, C_in, H, W = x.shape
  C_out, _, kH, kW = weight.shape
  patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
  print(f"{patches.shape=}")
  out = torch.einsum("bihwjk, oijk -> bohw", patches, weight)
  if bias is not None:
    out += bias.view(1, -1, 1, 1)
  return out



In [5]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

patches.shape=torch.Size([1, 3, 6, 6, 3, 3])
Output: torch.Size([1, 16, 6, 6])
patches.shape=torch.Size([1, 3, 6, 6, 3, 3])
Match: True


In [6]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
patches.shape=torch.Size([1, 3, 6, 6, 3, 3])
  ✅ [1/5] Output shape (1.1ms)
patches.shape=torch.Size([2, 3, 6, 6, 3, 3])
  ✅ [2/5] Matches F.conv2d (106.2ms)
patches.shape=torch.Size([1, 1, 5, 5, 3, 3])
  ✅ [3/5] With padding (8.3ms)
patches.shape=torch.Size([1, 1, 3, 3, 3, 3])
  ✅ [4/5] With stride (1.5ms)
patches.shape=torch.Size([1, 1, 2, 2, 3, 3])
  ✅ [5/5] Gradient flow (23.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (140.9ms total)
  Progress saved. Run status() to see your dashboard.



In [2]:
from torch_judge import hint
hint('conv2d')


💡 Hint for 2D Convolution:
   Extract patches using unfold or nested loops. For each output position, sum(patch * kernel). Support stride and padding (zero-pad with F.pad).

